# Librerías y Módulos

In [21]:
import sys

# Obtener la lista de kernels
kernels = !jupyter kernelspec list

# Mostrar la lista de kernels para referencia
print(kernels)

# Buscar si 'mi_entorno' está en la lista de kernels
kernel_name = "mi_entorno"
kernel_found = False

for line in kernels:
    if kernel_name in line:
        kernel_found = True
        break

if not kernel_found:
    print(f'Kernel debe ser: {kernel_name}')
    #sys.exit()

print(f'Kernel correcto: {kernel_name}')


['"jupyter" no se reconoce como un comando interno o externo,', 'programa o archivo por lotes ejecutable.']
Kernel debe ser: mi_entorno
Kernel correcto: mi_entorno


In [22]:
sys.path.insert(0, '')
from Clase_Valor import Valor
from Transversal import leer_activos, leer_parametros, leer_configuracion

## Librerías

In [23]:
import pandas as pd
import numpy as np
import datetime as dt
import itertools
import os
import matplotlib.pyplot as plt
import yfinance as yf

import pickle
import matplotlib.pyplot as plt
import tqdm
import time

from sklearn.model_selection import train_test_split

#import import_ipynb # permite importar módulos ipynb
import warnings
warnings.filterwarnings("ignore")

import mip

# Parámetros

In [24]:
carpeta_input, cofre, seguimiento, saving_step = leer_parametros()
output_level = leer_configuracion()

# Lectura de activos

In [25]:
df_activos = leer_activos(carpeta_input)

Small batch 240812


# Funciones

In [26]:
def pickle_act(file_name, variable = None, mode = 'open', eliminar_si_problemas = False):
    
    """
    Guarda o carga una variable utilizando la biblioteca pickle.

    Parameters:
        - file_path (str): La ruta al archivo pickle.
        - variable: La variable a guardar (si mode='save') o None (si mode='open').
        - mode (str): 'save' para guardar la variable, 'open' para cargar la variable.

    Returns:
        La variable cargada si mode='open' o None si mode='save'.
    """
    
    dic_mode = {'save': 'wb', 'open': 'rb'}
    
    #while True:
    #    try:
    with open(f'{file_name}.pkl', dic_mode[mode]) as file:
        if mode == 'save':
            pickle.dump(variable, file)
            return None
        else:
            #print('file en funciones transversales', f'{file_name}.pkl')
            if eliminar_si_problemas:
                try:
                    variable = pickle.load(file)
                except:
                    os.remove(f'{file_name}.pkl')
                    return pd.DataFrame()
            else:
                variable = pickle.load(file)
            return variable

# Clase Red Neuronal

In [27]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from keras.initializers import RandomUniform

In [28]:
class Callback(tf.keras.callbacks.Callback): # Esta clase impide que en el entrenamiento de la FFNN, se muestren los resultados de cada epoch (model.fit)
    SHOW_NUMBER = 30
    counter = 0
    epoch = 0

    def on_epoch_begin(self, epoch, logs = None):
        self.epoch = epoch

    def on_train_batch_end(self, batch, logs = None):
        if self.counter == self.SHOW_NUMBER or self.epoch == 1:
            None
            #print('Epoch: ' + str(self.epoch) + ' loss: ' + str(logs['loss']))
            if self.epoch > 1:
                self.counter = 0
        self.counter += 1

In [29]:
def definir_arquitectura(str_arquitectura): # convierte un str de arquitectura a una matriz de arquitectura
    arquitectura = []
    if str_arquitectura == "0":
        return arquitectura
    
    lista_neuronas = str_arquitectura.split(',')

    for neurona in lista_neuronas:
        arquitectura.append([neurona.split('_')[0], int(neurona.split('_')[1])])
    
    return arquitectura

In [30]:
def generar_matriz_transformacion(lista_representativa, eliminar):
    # Solo permite eliminar filas y columnas
    vector_rep = np.array(lista_representativa)
    #  T es una mariz de ceros, de dimensiones vector_rep.sum() [cantidad de capos compartidos, interseccion] x len(campos_base_x)
    T = np.zeros((vector_rep.sum(), len(vector_rep)))

    i, j = 0, 0
    for k in range(len(vector_rep)):
        if vector_rep[k] == 1:
            T[i, j] = 1
            i, j = i + 1, j + 1
        else:
            j += 1

    if eliminar == 'columnas': # Se devuelve la matriz transpuesta
        T = T.T
    return T


def generar_matriz_transformacion_estructurada(lista_representativa):
    # Permite agregar filas y columnas random
    
    vector_rep = np.array(lista_representativa)
    T = np.zeros((len(vector_rep), vector_rep.sum()))
    T.shape

    i, j = 0, 0
    for k in range(len(vector_rep)):
        if vector_rep[k] == 1:
            T[i, j] = 1
            i, j = i + 1, j + 1
        else:
            for j2 in range(vector_rep.sum()):
                # agregar un random uniform entre -1 y 1
                T[i, j2] = np.random.uniform(-1, 1)
            i += 1

    return T
    

In [31]:
def estandarizar(df):
    df_estandarizacion = pd.DataFrame()
    for c in list(set(df.columns) - {'VALOR', 'DATE', 'PREDICT'}):
        promedio, desvest = df[c].mean(), df[c].std()
        df_estandarizacion_new = pd.DataFrame({'CAMPO': [c], 'PROMEDIO': [promedio], 'DESVIACION': [desvest]})
        df_estandarizacion = pd.concat([df_estandarizacion, df_estandarizacion_new])
        df[c] = (df[c] - promedio) / desvest
    return df, df_estandarizacion

def normalizar(df):
    df_normalizacion = pd.DataFrame()
    for c in list(set(df.columns) - {'VALOR', 'DATE', 'PREDICT'}):
        n_min, n_max = df[c].min(), df[c].max()
        df_normalizacion_new = pd.DataFrame({'CAMPO': [c], 'MIN': [n_min], 'MAX': [n_max]})
        df_normalizacion = pd.concat([df_normalizacion, df_normalizacion_new])
        df[c] = (df[c] - n_min) / (n_max - n_min)
    return df, df_normalizacion


In [32]:
class Red_Neuronal():

    def __init__(self, str_arquitectura, campos_input, cofre, seguimiento, output_level, metrica, df_activos, herencia = False, nombre_clase_heredada = None, cambio = None, dic_cambios = None, campos_input_heredados = None): 
    
        # Se guardan en el init, los atributos iniciales del objeto
        self.str_arquitectura = str_arquitectura # arquitectura asociada a la red neuronal 
        #self.nombre = nombre # nombre de la red neuronal
        self.cofre = f'{cofre}Red_Neuronal/' # donde se guardan y rescatan las redes, con su info actualizada
        self.seguimiento = seguimiento
        self.output_level = output_level
        self.metrica = metrica
        self.raw_x = pd.DataFrame({'DATE': [], 'NAME': []})
        #self.campos_input = campos_input
        #self.campos_output = campos_output
        self.dic_data = {}
        self.best_test_loss = float('inf')
        self.herencia = herencia
        self.nombre_clase_heredada = nombre_clase_heredada
        self.cambio = cambio
        self.dic_cambios = dic_cambios
        self.df_activos = df_activos
        
        #print('Activos seleccionados 0')
        #display(self.df_activos)
        
        self.campos_input_heredados = campos_input_heredados
        #self.campos_input_heredados = campos_input
        #self.epochs = epochs
        
        self.campos_output = f'Y_{self.metrica}_{self.output_level}'
        
        self.inicializar(campos_input, self.campos_output)
        self.modo_arquitectura = self.rescatar() # Rescata el objeto (y toda su información) si existe
        
        #print('Activos seleccionados')
        #display(self.df_activos)
        return None
                
    def rescatar(self):
        if f'{self.nombre}.pkl' not in os.listdir(self.cofre):
            #print('Rescatar')
            self.arquitectura = definir_arquitectura(self.str_arquitectura) # Se define la arquitectura con su nomeclatura
            #print('NAME REV')
            #print(self.str_arquitectura, self.arquitectura)
            
            if self.herencia: # Si se puede, se rescatan los datos del objeto heredado
                print('SI HERENCIA FFNN')
                valor_cargado_heredado = pickle_act(f'{self.cofre}{self.nombre_clase_heredada}') # De lo contrario, se lee el objeto guardado y se rescatan sus atributos
                for key, value in vars(valor_cargado_heredado).items(): # vars contiene los atributo y sus valores como diccionario (str, obj) vars = {'x': valor de x, 'y': valor de y}
                    if key in ['herencia', 'nombre_clase_heredada', 'cambio', 'dic_cambios', 'str_arquitectura', 'cofre', 'seguimiento', 'output_level', 'arquitectura', 'campos_input', 'campos_input_heredados', 'df_activos']: # atributos no heredados
                        continue
                    elif key == 'model':
                        setattr(self, 'model_base', value) # Aquí, model se renombra como model base
                    #elif key == 'campos_input':
                    #    setattr(self, 'campos_input_heredados', value)
                    # si se heredan! dic_data, model 
                    setattr(self, key, value) # setattr(objeto, atributo, valor) -> objeto.atributo = valor, actúa sobre la clase self, recibe un key (str) y un value (obj) y los asigna a la clase como atributos: self.key = value...es similar a usar un globals(), pero en una clase
                #print('NAME REV2')
                #print(self.str_arquitectura, self.arquitectura)
                
                self.cambiar_ffnn() # El cambio solo existe si hay una herencia, y la nueva red no existe
            else:    
                self.crear_ffnn() # Crea la estructura inicial de la red
            return 'nueva' # Si no se encuentra, no se pueden rescatar los atributos
        
        #print('RESCATE!!')
        valor_cargado = pickle_act(f'{self.cofre}{self.nombre}') # De lo contrario, se lee el objeto guardado y se rescatan sus atributos (todos)
        for key, value in vars(valor_cargado).items(): # vars contiene los atributo y sus valores como diccionario (str, obj) vars = {'x': valor de x, 'y': valor de y}
            if key in ['df_activos']: # atributos no heredados
                continue
            setattr(self, key, value) # setattr(objeto, atributo, valor) -> objeto.atributo = valor, actúa sobre la clase self, recibe un key (str) y un value (obj) y los asigna a la clase como atributos: self.key = value...es similar a usar un globals(), pero en una clase
        return 'rescate'
    
    
    def guardar(self):
        pickle_act(f'{self.cofre}{self.nombre}', variable = self, mode = 'save')
        return None
    
    def crear_ffnn(self, loss = 'mean_squared_error', optimizer = 'adam', metrics = ['accuracy']):
        
        dic_metricas = {'Rendimiento': None, 'Varianza': 'relu'}
        # Inicializador con valores entre 0 y 1
        initializador = RandomUniform(minval = 0, maxval = 1)
    
        # Crea un modelo secuencial
        self.model = keras.Sequential()
        
        if len(self.arquitectura) == 0: # Sin hidden layers
            self.model.add(layers.Dense(self.output_dim, activation = None, input_shape = (self.input_dim,)))

        else:
            for i, layer in enumerate(self.arquitectura):
                #print('NUEVA CAPA', i, layer)
                if i == 0:
                    self.model.add(layers.Dense(layer[1], activation = layer[0], input_shape = (self.input_dim,), kernel_initializer = initializador)) # Capa de entrada
                else:
                    self.model.add(layers.Dense(layer[1], activation = layer[0], kernel_initializer = initializador)) # Cualquier otra capa
            
        # Agrega la capa de salida, con y_test.shape[1] neuronas y sin activación (para que quede libre, y no restringir el número a un no negativo (por ejemplo)
        self.model.add(layers.Dense(1, activation = dic_metricas[self.metrica])) # Rend puede ser negativo, Var no
        
        # Compliación y definición
        self.model.compile(loss = loss, optimizer = optimizer, metrics = metrics)
        
        #print('OK estructura FFNN creada')
        
        """
        for i in range(len(self.arquitectura) + 1):
            weights0_origen, biases0_origen = self.model.layers[i].get_weights()
            print('Capa estr', i)
            print('W estr', weights0_origen.shape)
            print('b estr', biases0_origen.shape)
            print(weights0_origen)
            print(biases0_origen)
        """

        #print('\n\n\n')
        return None 
    
    # cambiar_ffnn(self, cambio, capa_seleccionada, delta_n_neurs_new)
    def cambiar_ffnn(self):
        # Aqui los cambios
        #print('CAMBIAR FFNN')
        #print(self.cambio)
        
        #print('NAME REV3')
        #print(self.str_arquitectura, self.arquitectura)
        
        # 1.3 Agregar neuronas
        if self.cambio == '1_agregar_neuronas':
            
            #print('CREAR', self.arquitectura)
            self.crear_ffnn()
            
            capa_seleccionada = self.dic_cambios['capa_seleccionada']
            delta_n_neurs_new = self.dic_cambios['delta_n_neurs_new']
            #### 1.3.1 matriz anterior
            weights0_origen, biases0_origen = self.model_base.layers[capa_seleccionada].get_weights()
            #weights0_origen.shape, biases0_origen.shape

            # a weights0_origen, se le hace un hstack random
            filas_add, cols_add = weights0_origen.shape[0], delta_n_neurs_new
            A = np.random.rand(filas_add, cols_add)
            weights0_nuevo = np.hstack([weights0_origen, A]) # Se añaden las columnas nuevas

            # a biases0_origen, se le hace un hstack random
            A = np.random.rand(cols_add)
            biases0_nuevo = np.hstack([biases0_origen, A])

            self.model.layers[capa_seleccionada].set_weights([weights0_nuevo, biases0_nuevo]) # Los setea en la red neuronal"""

            #### 1.3.2 matriz siguiente
            weights1_origen, biases1_origen = self.model_base.layers[capa_seleccionada + 1].get_weights()
            weights1_origen.shape, biases1_origen.shape

            # a weights0_origen, se le hace un vstack random
            filas_add, cols_add = delta_n_neurs_new, weights1_origen.shape[1]
            A = np.random.rand(filas_add, cols_add)
            weights1_nuevo = np.vstack([weights1_origen, A]) # Se añaden las columnas nuevas

            self.model.layers[capa_seleccionada + 1].set_weights([weights1_nuevo, biases1_origen]) # Los setea en la red neuronal"""
            
            #print('Nueva estructura ajustada')
        
        elif self.cambio == '2_eliminar_neuronas':
                        
            self.crear_ffnn()
            
            capa_seleccionada = self.dic_cambios['capa_seleccionada'] 
            delta_n_neurs_new = self.dic_cambios['delta_n_neurs_new']
            dic_betas = self.dic_cambios['dic_betas']

            lista_betas_seleccion = dic_betas[capa_seleccionada] # errores imputados a las neuronas en las capas
            
            #print('dic betas', dic_betas)
            df_betas_seleccion = pd.DataFrame(lista_betas_seleccion, columns = ['BETA']) # Se seleccionan las neuronas que serán eliminadas
            df_betas_seleccion['NEURONA'] = df_betas_seleccion.index
            df_betas_seleccion['BETA_ABS'] = np.abs(df_betas_seleccion['BETA'])
            df_betas_seleccion = df_betas_seleccion.sort_values('BETA_ABS', ascending = False).reset_index(drop = True) # Se eligen las neuronas con betas más altos en valr abs
            n_neurs_a_eliminar = abs(self.dic_cambios['delta_n_neurs_new'])

            df_betas_seleccion = df_betas_seleccion.head(n_neurs_a_eliminar)
            neurs_a_eliminar = list(df_betas_seleccion['NEURONA'].unique())
            neurs_a_eliminar.sort()

            lista_representativa = [] # Lista representativa: 0 si es una neurona a eliminar y 1 si es una neurona a conservar
            for i in range(len(lista_betas_seleccion)):
                if i in neurs_a_eliminar:
                    lista_representativa.append(0)
                else:
                    lista_representativa.append(1)

            Ty = generar_matriz_transformacion(lista_representativa, eliminar = 'columnas') # matriz de transformación para eliminar columnas
            Tx = generar_matriz_transformacion(lista_representativa, eliminar = 'filas') 

            #### 1.3.1 matriz anterior (eliminar columnas)
            weights0_origen, biases0_origen = self.model_base.layers[capa_seleccionada].get_weights()  # cambiar por self.model.layers[capa_seleccionada].get_weights()

            weights0_nuevo = weights0_origen @ Ty # cambios en W (se eliminan cols, transformando por derecha)
            biases0_nuevo = Tx @ biases0_origen # cambios en b (se eliminan filas, transformando por izquierda)
            self.model.layers[capa_seleccionada].set_weights([weights0_nuevo, biases0_nuevo]) # seteo de las nuevas configuraciones

            #### 1.3.2 matriz siguiente
            weights1_origen, biases1_origen = self.model_base.layers[capa_seleccionada + 1].get_weights()

            weights1_nuevo = Tx @ weights1_origen # cambios en W (se eliminan filas, transformando por izquierda)
            # biases1_origen se mantiene
            self.model.layers[capa_seleccionada + 1].set_weights([weights1_nuevo, biases1_origen]) # seteo de las nuevas configuraciones
            
        elif self.cambio == '3_eliminar_capa': # ver hojas 16 & 17

            self.crear_ffnn()
            
            capa_seleccionada = self.dic_cambios['capa_seleccionada'] # cambiar por self.dic_cambios
            delta_n_neurs_new = self.dic_cambios['delta_n_neurs_new']
            
            for i in range(len(self.arquitectura) + 2): # +1 por capa de salida + 1 por capa de arquitectura original
                weights0_origen, biases0_origen = self.model_base.layers[i].get_weights()
                if i < capa_seleccionada:
                    self.model.layers[i].set_weights([weights0_origen, biases0_origen]) # se mantiene
                elif (i == capa_seleccionada) or (i == capa_seleccionada + 1): # se conserva el random generado (se pierde la matriz antecesora y sucesora de la capa)
                    None
                else:
                    self.model.layers[i - 1].set_weights([weights0_origen, biases0_origen]) # se mantiene
        
        elif self.cambio == '3_agregar_capa': # Ver hojas 17 & 18
            self.crear_ffnn()
            #{'capa_seleccionada': 3, 'neuronas_seleccionadas': 4, 'funcion_seleccionada': 'relu'}
            capa_seleccionada = self.dic_cambios['capa_seleccionada']
            for i in range(len(self.arquitectura) + 1): # +1 por capa de salida 
                if i < capa_seleccionada:
                    weights0_origen, biases0_origen = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                    self.model.layers[i].set_weights([weights0_origen, biases0_origen]) # se reemplazan en la nueva red directamente
                elif (i == capa_seleccionada) or (i == capa_seleccionada + 1): # se conserva el random generado (se pierde la matriz antecesora y sucesora de la capa)
                    None
                else:
                    weights0_origen, biases0_origen = self.model_base.layers[i - 1].get_weights()
                    self.model.layers[i].set_weights([weights0_origen, biases0_origen])
        
        elif (self.cambio == '4_cambio_funcion') or (self.cambio == '5_cambio_funcion'):
            
            self.crear_ffnn() # Se crea la nueva estructura con la nueva función de activación
            
            for i in range(len(self.arquitectura) + 1): # +1 por capa de salida 
                weights0_origen, biases0_origen = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                self.model.layers[i].set_weights([weights0_origen, biases0_origen]) # se reemplazan en la nueva red directamente
        
        elif self.cambio == '6_cambios_inputs_agregar':
            
            # AGREGAR!!!
            self.input_dim, self.output_dim = len(self.campos_input), len(self.campos_output)
            self.crear_ffnn() # Se crea la nueva estructura
                        
            lista_representativa = [] # Lista representativa: 0 si es una neurona nueva (se generará un random en la función generar_matriz_transformacion_estructurada) y 1 si es una neurona a conservar
            for elemento in self.campos_input:
                if elemento in self.campos_input_heredados:
                    lista_representativa.append(1)
                else:
                    lista_representativa.append(0)

            Tx = generar_matriz_transformacion_estructurada(lista_representativa)
            
            # Cambio en las matrices de input
            for i in range(len(self.arquitectura) + 1):
                weights, biases = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                if i == 0:
                    weights = Tx @ weights # cambios en W (se eliminan filas, transformando por izquierda)
                self.model.layers[i].set_weights([weights, biases]) # se reemplazan en la nueva red directamente
            
                
        elif self.cambio == '7_cambios_inputs_eliminar':
            
            self.input_dim, self.output_dim = len(self.campos_input), len(self.campos_output)
            self.crear_ffnn() # Se crea la nueva estructura
            
            lista_representativa = [] # Lista representativa: 0 si es ya no existe y 1 si es una neurona a conservar
            for elemento in self.campos_input_heredados:
                if elemento in self.campos_input:
                    lista_representativa.append(1)
                else:
                    lista_representativa.append(0)
            
            Tx = generar_matriz_transformacion(lista_representativa, 'filas')
            
            # Cambio en las matrices de input
            for i in range(len(self.arquitectura) + 1):
                weights, biases = self.model_base.layers[i].get_weights() # se obtienen los W-b de la red heredada
                if i == 0:
                    weights = Tx @ weights # cambios en W (se eliminan filas, transformando por izquierda)
                self.model.layers[i].set_weights([weights, biases]) # se reemplazan en la nueva red directamente

        return None
                
    def inicializar(self, campos_input, campos_output): # Si la red no fue rescatada, entonces se leen algunos parámetros
        #print('Inicializar')
        
        #print('campos_input', campos_input)
        # Estandarización de campos input y output
        if campos_input == None:
            self.campos_inputs_default() # Se generan los campos  inputs por  default: Todos los existentes en los valores que están en df_activos
        elif type(campos_input) == str:
            self.campos_input = campos_input.split(',')
        else:
            self.campos_input = campos_input
        self.campos_input.sort()
        
        self.campos_output = campos_output.split(',')
        self.campos_output.sort()
        
        self.str_campos_input, self.str_campos_output = ','.join(self.campos_input),  ','.join(self.campos_output)
        self.input_dim, self.output_dim = len(self.campos_input), len(self.campos_output)
        
        #print(self.campos_input, self.campos_output)
        #print('dims', self.input_dim, self.output_dim)
        
        # Reconoce si existe, o crea un id nuevo
        if 'df_info_ffnn.pkl' not in os.listdir(self.seguimiento):
            self.df_info_ffnn = pd.DataFrame(columns = ['str_arquitectura', 'campos_input', 'campos_output', 'nombre'])
            #return None
        else:
            self.df_info_ffnn = pickle_act(f'{self.seguimiento}df_info_ffnn') # Se lee si existe
            
        # Asignación de nombre...se rescata si existe, de lo contrario se asigna uno nuevo
        df_info_ffnn_filtrado = self.df_info_ffnn[(self.df_info_ffnn['str_arquitectura'] == self.str_arquitectura) & (self.df_info_ffnn['campos_input'] == self.str_campos_input) & (self.df_info_ffnn['campos_output'] == self.str_campos_output)].reset_index(drop = True) 
        
        if len(df_info_ffnn_filtrado) == 0: # asignación de nombre
            self.nombre = 'FFNN_'+ str(len(self.df_info_ffnn) + 1) # Se asigna un nombre con un id
            # Actualización de df info y guardado
            new_row = pd.DataFrame({'str_arquitectura': [self.str_arquitectura], 'campos_input': [self.str_campos_input], 'campos_output': [self.str_campos_output], 'nombre': [self.nombre]})
            self.df_info_ffnn = pd.concat([self.df_info_ffnn, new_row], axis = 0).reset_index(drop = True)
            pickle_act(f'{self.seguimiento}df_info_ffnn', variable = self.df_info_ffnn, mode = 'save') # Se guarda, solo si se agrega un caso nuevo
        else:
            self.nombre = df_info_ffnn_filtrado['nombre'][0]

        return None
    
    
    def campos_inputs_default(self):
        
        #print('campos_inputs_default')
        cofre0 = '/'.join(self.cofre.split('/')[:-2]) + '/'
        set_campos_inputs = set()
        for i in range(len(self.df_activos)):
            simbolo, nombre = self.df_activos.loc[i]
            # Si no existe el objeto, se crea
            valor = Valor(simbolo, nombre, cofre0) # 
            if len(valor.raw_x) == 0: # No hay datos que aportar
                continue
            set_campos_inputs_new = set([campo for campo in valor.raw_x['NAME'].unique() if campo[:2] != "Y_"])
            #print('set_campos_inputs_new')
            #print(set_campos_inputs_new)
            set_campos_inputs = set_campos_inputs.union(set_campos_inputs_new)
        self.campos_input = list(set_campos_inputs)
        
        #self.campos_input = self.campos_input[:10] # fijado: cambiar!!
        
        return None
    
    def construir_matrices(self):
        
        while True:
        
            # Matrices de input y de output
            #print('Campos inputs en construir matrices')
            #print(self.campos_input)
            #print(self.campos_output)
            # output_level = 10 # 10 días para este ejemplo (esta red neuronal concretamente)
            cofre0 = '/'.join(self.cofre.split('/')[:-2]) + '/'
            df_raw_info = pd.DataFrame()
            for i in range(len(self.df_activos)):
                simbolo, nombre = self.df_activos.loc[i]
                valor = Valor(simbolo, nombre, cofre0) # Si no existe el objeto, se crea 
                if len(valor.raw_x) == 0: # No hay datos que aportar
                    continue
                #print(simbolo, nombre, len(valor.raw_x))
                new_raw_x = valor.raw_x.copy()
                new_raw_x['VALOR'] = simbolo
                
                print('SIMBOLO', simbolo, new_raw_x['DATE'].min(), new_raw_x['DATE'].max())
                df_raw_info = pd.concat([df_raw_info, new_raw_x], axis = 0)
            

            #display(df_raw_info[df_raw_info['X'].isna()])
            #sys.exit('df raw info nan')
            df_raw_info = df_raw_info[['VALOR', 'DATE', 'NAME', 'X']]
            #df_raw_info.head()
            
            #print('Name en raw info')
            #print(df_raw_info['NAME'].unique())
            df_raw_info = df_raw_info[(df_raw_info['NAME'].isin(self.campos_input)) | (df_raw_info['NAME'].str[:2] == 'Y_')] # Se seleccionan solo los campos de input que están declarados en campos input
            #sys.exit('Revisar esto (deberian seleccionarse solo los campos inputs)!')

            # Primero, se separan los inputs de los outputs
            df_raw_info['NATURALEZA'] = np.where(df_raw_info['NAME'].str[:2] == 'Y_', 'Y', 'X')
            df_raw_y = df_raw_info[df_raw_info['NATURALEZA'] == 'Y'].reset_index(drop = True)
            df_raw_x = df_raw_info[df_raw_info['NATURALEZA'] == 'X'].reset_index(drop = True)

            df_X = df_raw_x.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()
            df_Y = df_raw_y.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()

            df = df_X.merge(df_Y, on = ['VALOR', 'DATE'], how = 'outer')
        
        #df.to_csv(f'dfX_{self.nombre}.csv', index = False, decimal = ',', sep = ';')
        #sys.exit('dfX.csv')
        
            aprobado = True
            for c in list(set(df.columns) - {'DATE', 'VALOR'}):
                if len(df[df[c].isna()]) > 0:
                    aprobado = False
                    print(f'El campo {c} tiene valores nulos')
                    display(df[df[c].isna()])
                    sys.exit('Salida por campos nulos')
            if aprobado:
                break
            print('B. Espera de completitud de valores en Valor.py: Espera de 30s.')
            time.sleep(30)
        
        """
        for c in list(set(df.columns) - {'DATE', 'VALOR'}):
            if len(df[df[c].isna()]) > 0:
                print(c)
                display(df[df[c].isna()])
                sys.exit('Error en campo')
        """

        # Limpieza de df
        self.lista_campos_output = list(set(df_Y.columns) - {'DATE', 'VALOR'})
        set_delta_dates = set()
        for c in self.lista_campos_output:

            delta = c.split('_')[-1]
            set_delta_dates.add(delta)
        #lista_delta_dates = list(set_delta_dates)
        
        df['PREDICT'] = np.where((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0), True, False)
        #print('lista_delta_dates', lista_delta_dates)
        df, self.df_normalizacion = normalizar(df)

        #df.to_csv(f'dfX_v2_{self.nombre}.csv', index = False, decimal = ',', sep = ';')
        
        #print('La idea es normalizar esta BD')
        #sys.exit(f'Salida para revisión de dfX_v2_{self.nombre}.csv')
        
        delta = self.output_level

        print(f'Y_Rendimiento_{delta}', f'Y_Varianza_{delta}')            
        self.df_predict = df[df['PREDICT']].reset_index(drop = True) # Se aislan los datos para después hacer el predict
        self.df_predict = self.df_predict.drop(columns = ['PREDICT'])
        if len(self.df_predict) == 0:
            sys.exit('Predict sin datos')
        else:
            None
            #print(self.nombre)
            #print(self.df_predict)
        df = df[~df['PREDICT']].reset_index(drop = True) # Se excluyen los casos en los que la varianza y el rend de un día, son 0, para el mismo delta
        df = df.drop(columns = ['PREDICT'])
    
        
        """
        for delta in lista_delta_dates:
            print(f'Y_Rendimiento_{delta}', f'Y_Varianza_{delta}')            
            self.df_predict = df[((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se aislan los datos para después hacer el predict
            if len(self.df_predict) == 0:
                sys.exit('Predict sin datos')
            else:
                print(self.nombre)
                print(self.df_predict)
            df = df[~((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se excluyen los casos en los que la varianza y el rend de un día, son 0, para el mismo delta
        """
        
        if self.output_level not in self.dic_data:
            self.dic_data[self.output_level] = df
        
        return None
    
    def split_data(self, test_size = 0.2): # Como método propio, para poder generar sampleos aleatorios libremente
        
        df = self.dic_data[self.output_level].copy()
        df = df.fillna(0) # Provisorio, para activos que están incompletos con sus datos históricos
        df = df.drop(columns = ['DATE', 'VALOR'])
        Y = df[f'Y_{self.metrica}_{self.output_level}'] # Una sola métrica de output ya que si es varianza,hay que asegurar que sea >= 0
        X = df.drop(columns = self.lista_campos_output)
        x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = test_size) # Split aleatorio de los datos
        x_train, x_test, y_train, y_test = x_train.values, x_test.values, y_train.values, y_test.values # dt to matriz numpy
        return x_train, x_test, y_train, y_test
    
    def matrices_predict(self): # nuevo 240614
        df_x_predict = self.df_predict.drop(columns = self.lista_campos_output) 
        df_x_predict_values = df_x_predict.drop(columns = ['DATE', 'VALOR'])
        x_predict = df_x_predict_values.values
        return df_x_predict, x_predict
    
    def predict_model(self, x_predict):
        y_predict = self.model.predict(x_predict)
        return y_predict
    
    def train_model(self, x_train, x_test, y_train, y_test, batch_size = 32, epochs = 5, n_min = 200, plotear = False): # n_min: cuantos epochs sin superar el best son necesarios para stop
        
        #print('EN TRAIN MODEL\n\n\n\n\n')
        if not plotear:
            e, k = 0, 0
            while True:
                #print('K', k)

                self.model.fit(x_train, y_train, epochs = 1, batch_size = batch_size, validation_data = (x_test, y_test), callbacks = [Callback()], verbose = 0) # Entrenamiento del modelo
                # batch_size: cuantos datos juntos se entrenan a la vez, antes de actualizar los parámetros
                # verbose = 0: muestra menos información de output
                # self.train_loss, self.train_accuracy = self.model.evaluate(x_train, y_train) # Evaluación de test y obtención de performance de testeo
                #print('W-b en train')
                [w, b] = self.model.layers[0].get_weights()
                #print(w)
                #print(b)
                self.test_loss, self.test_accuracy = self.model.evaluate(x_test, y_test) # Evaluación de test y obtención de performance de testeo
                #print(\test_loss, test_accuracy\, self.test_loss, self.test_accuracy)
                if self.test_loss < self.best_test_loss: # Se guardan los mejores registros y el mejor modelo para la predicción (si es que los parámetros mejoran)
                    self.best_test_loss = self.test_loss
                    self.best_test_accuracy = self.test_accuracy
                    self.best_model = self.model
                    k = 0 # reset
                k += 1 # contador de cuantos epochs van sin mejorar el best loss
                if k >= n_min: # criterio de salida
                    break
                
            return None
        
        # plotear = True
        df_plot = pd.DataFrame()
        for e in range(epochs):
            print('epoch', e)
            self.model.fit(x_train, y_train, epochs = 1, batch_size = batch_size, validation_data = (x_test, y_test), verbose = 0)
            
            [w, b] = self.model.layers[0].get_weights()
            #print('W-b en train')
            #print(w)
            #print(b)
            train_loss, train_accuracy = self.model.evaluate(x_train, y_train) # Evaluación de test y obtención de performance de testeo
            test_loss, test_accuracy = self.model.evaluate(x_test, y_test) # Evaluación de test y obtención de performance de testeo
            new_df = pd.DataFrame({'EPOCH': [e], 'TRAIN_LOSS': [train_loss], 'TRAIN_ACCURACY': [train_accuracy], 'TEST_LOSS': [test_loss], 'TEST_ACCURACY': [test_accuracy]})
            display(new_df)
            df_plot = pd.concat([df_plot, new_df], axis = 0)
        
        # plotea
        fig, ax = plt.subplots(1, 2, figsize = (15, 5))
        ax[0].plot(df_plot['EPOCH'], df_plot['TRAIN_LOSS'], label = 'TRAIN LOSS')
        ax[0].plot(df_plot['EPOCH'], df_plot['TEST_LOSS'], label = 'TEST LOSS')
        ax[0].legend()
        ax[0].set_title('LOSS')
        
        ax[1].plot(df_plot['EPOCH'], df_plot['TRAIN_ACCURACY'], label = 'TRAIN ACCURACY')
        ax[1].plot(df_plot['EPOCH'], df_plot['TEST_ACCURACY'], label = 'TEST ACCURACY')
        ax[1].legend()
        ax[1].set_title('ACCURACY')
        plt.show()
        
        self.df_plot = df_plot.copy()
        
        return None
        
        # plotear = True
        df_plot = pd.DataFrame()
        for e in range(epochs):
            self.model.fit(x_train, y_train, epochs = 1, batch_size = batch_size, validation_data = (x_test, y_test), verbose = 0)
            train_loss, train_accuracy = self.model.evaluate(x_train, y_train) # Evaluación de test y obtención de performance de testeo
            test_loss, test_accuracy = self.model.evaluate(x_test, y_test) # Evaluación de test y obtención de performance de testeo
            new_df = pd.DataFrame({'EPOCH': [e], 'TRAIN_LOSS': [train_loss], 'TRAIN_ACCURACY': [train_accuracy], 'TEST_LOSS': [test_loss], 'TEST_ACCURACY': [test_accuracy]})
            df_plot = pd.concat([df_plot, new_df], axis = 0)
        
        # plotea
        fig, ax = plt.subplots(1, 2, figsize = (15, 5))
        ax[0].plot(df_plot['EPOCH'], df_plot['TRAIN_LOSS'], label = 'TRAIN LOSS')
        ax[0].plot(df_plot['EPOCH'], df_plot['TEST_LOSS'], label = 'TEST LOSS')
        ax[0].legend()
        ax[0].set_title('LOSS')
        
        ax[1].plot(df_plot['EPOCH'], df_plot['TRAIN_ACCURACY'], label = 'TRAIN ACCURACY')
        ax[1].plot(df_plot['EPOCH'], df_plot['TEST_ACCURACY'], label = 'TEST ACCURACY')
        ax[1].legend()
        ax[1].set_title('ACCURACY')
        plt.show()
        
        self.df_plot = df_plot.copy()
        
        return None
    
    def error_imputado_neuronas(self, x_train, y_train):
        dic_metricas = {'Rendimiento': None, 'Varianza': 'relu'}
        dic_funciones = {'sigmoid': lambda x: 1 / (1 + np.exp(-x)), None: lambda x: x, 'relu': lambda x: np.maximum(0, x), 'tanh': lambda x: np.tanh(x)}
        dic_derivadas = {'sigmoid': lambda x: x * (1 - x), None: lambda x: 1, 'relu': lambda x: 1 if x > 0 else 0, 'tanh': lambda x: 1 - x ** 2}

        # Algoritmo de obtención de datos
        
        #print(self.model.layers)
        #print(self.arquitectura)
        #for i in range(len(self.model.layers)):
        #    [w, b] = self.model.layers[i].get_weights()
        #    print(i, w.shape, b.shape)
        #sys.exit('Revisar 240524')

        dic_ecuaciones = {}

        # 0. Inicio
        x = x_train
        
        for capa, detalle in enumerate(self.arquitectura):
            [act_fun, n_neurs] = detalle
            #print(capa, act_fun, n_neurs)
            # self.model.layers[capa].get_weights()
            [w, b] = self.model.layers[capa].get_weights()
            if np.isnan(w[0][0]):
                sys.exit('W vacío')
            z = x @ w + b 
            a = dic_funciones[act_fun](z)
                        
            dic_ecuaciones[capa] = {'z': z, 'a': a, 'x': x, 'w': w, 'b': b, 'act_fun': act_fun}
            x = a
            

        # Capa salida
        capa, act_fun, n_neurs = capa + 1, dic_metricas[self.metrica], 1
        #print(capa, act_fun, n_neurs)
        [w, b] = self.model.layers[capa].get_weights()
        z = x @ w + b 
        a = dic_funciones[act_fun](z)
        dic_ecuaciones[capa] = {'z': z, 'a': a, 'x': x, 'w': w, 'b': b, 'act_fun': act_fun}

        # 1. Obtener responsabilidad de cada neurona
        dic_deltas = {}
        # 1.1 Para la ultima capa
        # dL = (a - y) * f'(z)

        a = dic_ecuaciones[capa]['a']
        z = dic_ecuaciones[capa]['z']
        y = y_train.reshape(-1, 1)
        
        derivada = np.vectorize(dic_derivadas[act_fun])(z)
        dL = (a - y) * derivada
        dic_deltas[capa] = dL

        # Para cualquier capa intermedia
        # dL = (dL @ w.T) * f'(z)

        while True:
            w = dic_ecuaciones[capa]['w']
            if capa > 0:
                act_fun_prev = dic_ecuaciones[capa - 1]['act_fun']
                z_prev = dic_ecuaciones[capa - 1]['z']
                a_prev = dic_ecuaciones[capa - 1]['a']
            dL_next = dic_deltas[capa]

            if capa == 0:
                derivada_prev = 1
            elif act_fun_prev == 'relu': # La derivada se aplica a x, y no a f(x)
                derivada_prev = np.vectorize(dic_derivadas[act_fun_prev])(z_prev)
            else:
                derivada_prev = np.vectorize(dic_derivadas[act_fun_prev])(a_prev) # ya que a = f(z) y las derivadas de sigm y tanh están definidas en función de f(z)

            dL = (dL_next @ w.T) * derivada_prev
            capa -= 1
            dic_deltas[capa] = dL
            
            #print('Capa y dL', capa, dL)
            
            if capa == -1: # capa 0 es la primera capa oculta
            #if capa == 0: # capa 0 es la primera capa oculta
                break

        # betas
        self.dic_betas = {}
        for capa in dic_deltas:
            beta = dic_deltas[capa].sum(axis = 0)
            self.dic_betas[capa] = beta
            #print(capa, beta)
        
        return None
    
    #print('C:\Users\mvaldiviad\OneDrive - Falabella\Escritorio\Proyectos Personales\Markowitz\0. Markowitz FFNN 240322 V1_traspaso a ALginvesting.ipynb')
    

# Clase Búsqueda Inteligente

In [33]:
def agregar_o_eliminar_inputs(campos_inputs, all_campos_inputs, modo, beta_inputs = []):
    print('campos_inputs', len(campos_inputs), modo)
    print(campos_inputs)
    
    dic_cambios = {}
    if modo == 'agregar':
        delta_opciones = all_campos_inputs - set(campos_inputs)
        delta_agr = len(delta_opciones)
        if delta_agr == 0:
            print('No se pueden agregar más campos')
            dic_cambios['continuar'] = False

        else:
            # elegir random entre 1 y delta_agr
            delta_agr_new = np.random.choice(range(1, delta_agr + 1), p = obtener_lista_probs(delta_agr))
            # elegir delta_agr_new campos random de delta_opciones
            campos_inputs_new = list(np.random.choice(list(delta_opciones), delta_agr_new, replace = False))
            
            print('\n\n\ndelta new', delta_agr_new)
            dic_cambios['continuar'] = True
            campos_inputs += campos_inputs_new
            dic_cambios['campos_inputs_new'] = campos_inputs
            
    else: # modo "eliminar"
        delta_eliminar = len(campos_inputs) - 1 # tiene que quedar al menos un campo de input
        if delta_eliminar == 0:
            print('No se pueden eliminar más campos')
            dic_cambios['continuar'] = False
        
        else:
            delta_eliminar_new = np.random.choice(range(1, delta_eliminar + 1), p = obtener_lista_probs(delta_eliminar))
            #campos_a_eliminar = list(np.random.choice(campos_inputs, delta_eliminar_new, replace = False))
            
            print('\n\n\ndelta_eliminar_new', delta_eliminar_new)
            if len(beta_inputs) == 0: # caso random, en caso de que no exista entrenamiento
                # crea una lista random uniforme (0,1) de tamaño len(campos_inputs)
                beta_inputs = np.random.rand(len(campos_inputs))

            #print('beta_inputs', beta_inputs)
            df_betas_seleccion = pd.DataFrame(beta_inputs, columns = ['BETA']) # Se seleccionan las neuronas que serán eliminadas
            df_betas_seleccion['IDX'] = df_betas_seleccion.index
            
            df_neurs = pd.DataFrame(campos_inputs, columns = ['NEURONA_INPUT'])
            df_neurs['IDX'] = df_neurs.index
            
            df_betas_seleccion = df_betas_seleccion.merge(df_neurs, on = 'IDX', how = 'left')
        
            df_betas_seleccion['BETA_ABS'] = np.abs(df_betas_seleccion['BETA'])
            df_betas_seleccion = df_betas_seleccion.sort_values('BETA_ABS', ascending = False).reset_index(drop = True) # Se eligen las neuronas con betas más altos en valr abs

            df_betas_seleccion = df_betas_seleccion.head(delta_eliminar_new)
            neurs_a_eliminar = list(df_betas_seleccion['NEURONA_INPUT'].unique())
            neurs_a_eliminar.sort()
            
            #display(df_betas_seleccion)
            
            for n in neurs_a_eliminar:
                campos_inputs.remove(n)
                
            dic_cambios['continuar'] = True
            dic_cambios['campos_inputs_new'] = campos_inputs # Solo se entrega delta_eliminar_new, ya que los inputs eliminados se eligen según el criterio de los betas
    
    return dic_cambios
            


In [34]:
def agregar_o_eliminar_neuronas(lista_n_neurs, str_arquitectura, modo):
    
    arquitectura = definir_arquitectura(str_arquitectura) # Se define la arquitectura con su nomeclatura
    # 1. Agregar neuronas

    dic_n_neurs = {n_neurs: i for i, n_neurs in enumerate(lista_n_neurs)}
    dic_n_neurs_inv = {i: n_neurs for i, n_neurs in enumerate(lista_n_neurs)}

    # 1.1 Se revisa que capas son potenciales para agregar neuronas
    dic_neurs_potenciales, dic_n_new = {}, {}
    max_n_neurs = max(lista_n_neurs)
    for capa, detalle in enumerate(arquitectura):
        [act_fun, n_neurs] = detalle
        #print(capa, act_fun, n_neurs)
        if n_neurs == max_n_neurs: # No se pueden agregar mas neuronas en esta capa
            continue
        id_posicion = dic_n_neurs[n_neurs]
        id_posicion_new = id_posicion + 1 # para agregar
        if modo == 'eliminar':
            id_posicion_new = id_posicion - 1 # para eliminar
        n_neurs_new = dic_n_neurs_inv[id_posicion_new] # cual es el n_neurs que viene en lista_n_neurs
        delta_neurs = n_neurs_new - n_neurs # cuantas neuronas se agregarían
        dic_neurs_potenciales[capa] = delta_neurs
        dic_n_new[capa] = n_neurs_new

    #print('dic_neurs_potenciales')
    #print(dic_neurs_potenciales)
    
    # 1.2 Elegir random una de las capas potenciales para agregar neuronas
    capa_opciones = list(dic_neurs_potenciales.keys())

    # elige uno al azar y equiprobable con random choice
    capa_seleccionada = np.random.choice(capa_opciones) # capa seleccionada
    delta_n_neurs_new = dic_neurs_potenciales[capa_seleccionada] # y cuantas neuronas se agregarán
    n_news = dic_n_new[capa_seleccionada] # cuantas neuronas en total tendrá la capa seleccionada

    # Nombre de la nueva arquitectura
    #print('Ante cualquier modificación definida, revisar antes que todo, si esa estructura ya existe y está guardada')
    str_arquitectura_new = str_arquitectura.split(',')
    str_arquitectura_new_capa = str_arquitectura_new[capa_seleccionada].split('_')
    str_arquitectura_new_capa = '_'.join([str_arquitectura_new_capa[0], str(int(str_arquitectura_new_capa[1]) + delta_n_neurs_new)])
    str_arquitectura_new = str_arquitectura_new[:capa_seleccionada] + [str_arquitectura_new_capa] + str_arquitectura_new[capa_seleccionada + 1:]
    str_arquitectura_new = ','.join(str_arquitectura_new)
    #print(str_arquitectura_new)

    dic_cambios = {'capa_seleccionada': capa_seleccionada, 'delta_n_neurs_new': delta_n_neurs_new, 'n_news': n_news}
    
    return str_arquitectura_new, dic_cambios

def agregar_nueva_capa(lista_n_neurs, str_arquitectura):
    
    arquitectura = definir_arquitectura(str_arquitectura) # Se define la arquitectura con su nomeclatura

    lista_n_neurs_new_layer = lista_n_neurs.copy()
    lista_n_neurs_new_layer.remove(0)

    # identificar etapa de crecimiento y decrecimiento

    n_capas = [arquitectura[i][1] for i in range(len(arquitectura))]
    dic_cambios_fase = {}
    for i in range(1, len(n_capas)):
        if n_capas[i] < n_capas[i - 1]:
            dic_cambios_fase['decrecimiento'] = i
            break

    for i in range(len(n_capas) - 2, -1, -1):
        if n_capas[i] < n_capas[i + 1]:
            dic_cambios_fase['crecimiento'] = i
            break

    dic_etapas = {}
    for i in range(len(n_capas)):
        if ('crecimiento' in dic_cambios_fase) and (i <= dic_cambios_fase['crecimiento']):
            dic_etapas[i] = 'crecimiento'
        elif ('decrecimiento' in dic_cambios_fase) and (i >= dic_cambios_fase['decrecimiento']):
            dic_etapas[i] = 'decrecimiento'
        else:
            dic_etapas[i] = 'estable'

    # COMBINACIONES: c-c (entre), c-e (>= n neurs capa c), c-d (no existe), e-e (>= n neurs capa e (cualquiera, las dos tienen lo mismo)), e-d (>= n neurs capa d), d-d (entre)
    # Puedo validar la arquitectura post...si existe un cambio != a los de arriba, está mal

    dic_neurs_potenciales = {}
    for j in range(len(arquitectura) + 1):
        if j == 0: # inicial (antes de la capa 0)
            lista_opciones = [k for k in lista_n_neurs_new_layer if k <= n_capas[0]] # casos <= n_neurs de capa inicial
        elif (j > 0) and (j != len(arquitectura)):
            sigla_cambio = dic_etapas[j - 1][0] + '-' + dic_etapas[j][0]
            if sigla_cambio == 'c-c':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j - 1] and k <= n_capas[j]]
            if sigla_cambio == 'c-e':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j - 1]]
            if sigla_cambio == 'c-d':
                sys.exit('Esta combinación no debería existir')
            if sigla_cambio == 'e-e':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j]]
            if sigla_cambio == 'e-d':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k >= n_capas[j]]
            if sigla_cambio == 'd-d':
                lista_opciones = [k for k in lista_n_neurs_new_layer if k <= n_capas[j - 1] and k >= n_capas[j]]
        else:
            lista_opciones = [k for k in lista_n_neurs_new_layer if k <= n_capas[j - 1]]
        
        dic_neurs_potenciales[j] = lista_opciones

    # Elegir que capa se agregará
    capa_opciones = list(dic_neurs_potenciales.keys())
    capa_seleccionada = np.random.choice(capa_opciones) # capa seleccionada

    neuronas_potenciales = dic_neurs_potenciales[capa_seleccionada]
    neuronas_seleccionadas = np.random.choice(neuronas_potenciales) # n neurs seleccionadas para la capa

    lista_funciones = ['sigmoid', 'relu', 'tanh']
    funcion_seleccionada = np.random.choice(lista_funciones)

    ###
    str_arquitectura_new = str_arquitectura.split(',')
    str_arquitectura_new_capa = '_'.join([funcion_seleccionada, str(neuronas_seleccionadas)])
    str_arquitectura_new = str_arquitectura_new[:capa_seleccionada] + [str_arquitectura_new_capa] + str_arquitectura_new[capa_seleccionada:]
    str_arquitectura_new = ','.join(str_arquitectura_new)

    #print(str_arquitectura_new)
    dic_cambios = {'capa_seleccionada': capa_seleccionada, 'neuronas_seleccionadas': neuronas_seleccionadas, 'funcion_seleccionada': funcion_seleccionada}

    return str_arquitectura_new, dic_cambios

def cambiar_funcion(str_arquitectura, modo):
    
    arquitectura = definir_arquitectura(str_arquitectura) # Se define la arquitectura con su nomeclatura
    
    # 1.2 Elegir random una de las capas potenciales para agregar neuronas
    capa_opciones = list(range(len(arquitectura)))

    # elige uno al azar y equiprobable con random choice
    capa_seleccionada = np.random.choice(capa_opciones) # capa seleccionada
    
    funcion_seleccionada = arquitectura[capa_seleccionada][0]
    
    cambios_funcion = {'siguiente': {'tanh': 'relu', 'relu': 'sigmoid', 'sigmoid': 'tanh'},
                       'anterior': {'relu': 'tanh', 'sigmoid': 'relu', 'tanh': 'sigmoid'}}
    
    nueva_funcion = cambios_funcion[modo][funcion_seleccionada]
    
    str_arquitectura_new = str_arquitectura.split(',')
    str_arquitectura_new_capa = str_arquitectura_new[capa_seleccionada].split('_')
    str_arquitectura_new_capa = '_'.join([nueva_funcion, str_arquitectura_new_capa[1]])
    str_arquitectura_new = str_arquitectura_new[:capa_seleccionada] + [str_arquitectura_new_capa] + str_arquitectura_new[capa_seleccionada + 1:]
    str_arquitectura_new = ','.join(str_arquitectura_new)
    
    dic_cambios = {'nueva_funcion': nueva_funcion, 'modo': modo}

    return str_arquitectura_new, dic_cambios

def ajustar_str_arquitectura_eliminar_capa(str_arquitectura_new):
    capas = str_arquitectura_new.split(',')
    capas_new = []
    for capa in capas:
        if capa[-2:] == '_0':
            continue
        capas_new.append(capa)
    capas_new
    str_arquitectura_new = ','.join(capas_new)
    return str_arquitectura_new


In [35]:
def Probabilidad_Poisson(lambd, k):
    return (lambd ** k) * np.exp(-lambd) / np.math.factorial(k)


def obtener_lista_probs(n_max, factor = 0.3): # Factor -> lambda = factor * max_n
    df_probs = pd.DataFrame({'k': list(range(1, n_max + 1))})
    max_k = df_probs['k'].max()
    df_probs['Pr'] = df_probs['k'].apply(lambda x: Probabilidad_Poisson(max_k * factor, x))
    S = df_probs['Pr'].sum()
    df_probs['Pr'] = df_probs['Pr'] / S
    list_probs = list(df_probs['Pr'])
    return list_probs

In [36]:
class Busqueda_Inteligente():
    
    def __init__(self, nombre, metrica, output_level, cofre, lista_n_neurs, batch_size = 32, epochs = 50):
        
        n_movs = 7
        
        self.nombre = f'{nombre}_{metrica}_{output_level}'
        self.metrica = metrica
        self.output_level = output_level
        self.batch_size = batch_size
        self.epochs = epochs
        self.cofre = f'{cofre}Busqueda_Inteligente/'
        self.lista_n_neurs = lista_n_neurs
        self.cofre_base = cofre
        self.campos_input = None
        self.str_arquitectura = "tanh_4,relu_8,sigmoid_4" # arquitectura inicial (si no existe nada)
        self.inicial = True
        
        #print('df activos 0 en busqint')
        #display(df_activos)
        self.df_activos = df_activos
        self.best_test_loss_global = float('inf')
        self.dic_movimientos = {i: 0 for i in range(1, n_movs + 1)}
        self.obtener_all_campos_inputs() # obtiene cuantos campos inputs diferentes hay en total
        self.rescatar() # Si existe, se rescatan los atributos de la clase

        #print('df activos 1 en busqint')
        #display(self.df_activos)
        
        return None
    
    def rescatar(self):
        
        print(f'{self.nombre}.pkl')
        print(os.listdir(self.cofre))
        
        if f'{self.nombre}.pkl' not in os.listdir(self.cofre):
            return None # Si no se encuentra, no se pueden rescatar los atributos
    
        print(f'Rescate en {self.cofre}') # eliminar
        valor_cargado = pickle_act(f'{self.cofre}{self.nombre}') # De lo contrario, se lee el objeto guardado y se rescatan sus atributos
        for key, value in vars(valor_cargado).items(): # vars contiene los atributo y sus valores como diccionario (str, obj) vars = {'x': valor de x, 'y': valor de y}
            if key in ['df_activos']: # atributos no heredados
                continue
            setattr(self, key, value) # setattr(objeto, atributo, valor) -> objeto.atributo = valor, actúa sobre la clase self, recibe un key (str) y un value (obj) y los asigna a la clase como atributos: self.key = value...es similar a usar un globals(), pero en una clase
        return None
    
    def obtener_all_campos_inputs(self):
            
        print('campos_inputs_default en busq int')
        cofre0 = '/'.join(self.cofre.split('/')[:-2]) + '/'
        all_campos_inputs = set()
        for i in range(len(self.df_activos)):
            simbolo, nombre = self.df_activos.loc[i]
            # Si no existe el objeto, se crea
            valor = Valor(simbolo, nombre, cofre0) # 
            if len(valor.raw_x) == 0: # No hay datos que aportar
                continue
            set_campos_inputs_new = set([campo for campo in valor.raw_x['NAME'].unique() if campo[:2] != "Y_"])
            all_campos_inputs = all_campos_inputs.union(set_campos_inputs_new)
        self.all_campos_inputs = all_campos_inputs
        print('     len', len(all_campos_inputs))
        
        return None
    
    def ffnn_actual(self):
        self.ffnn = Red_Neuronal(self.str_arquitectura, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos)
        self.campos_input = self.ffnn.campos_input
        print(self.str_arquitectura)
        return None
    
    def train(self, n_min):
        self.ffnn.construir_matrices() 
        x_train, x_test, y_train, y_test = self.ffnn.split_data()
        
        self.ffnn.train_model(x_train, x_test, y_train, y_test, batch_size = self.batch_size, n_min = n_min, epochs = None, plotear = False) # plotear mientras!! no se guarda el entrenamiento, solo para validar
        self.best_test_loss_iter = self.ffnn.best_test_loss
        print('BEST TEST LOSS', self.ffnn.best_test_loss)
        self.ffnn.error_imputado_neuronas(x_train, y_train)
        return None

    def elegir_nueva_arquitectura(self, n_min):
        
        max_capas_permitidas = 5
        
        dic_selecciones = {1: 'agregar neuronas', 2: 'eliminar neuronas (o capa)', 3: 'agregar capa', 4: 'cambiar act_function', 5: 'cambiar act_function', 6: 'agregar inputs', 7: 'eliminar inputs'}
        # elegir nueva arquitectura (anclar arriba)
        n_movs = len(self.dic_movimientos)
        
        #dic_movimientos = {i: 0 for i in range(1, n_movs + 1)}

        min_value = min(list(self.dic_movimientos.values()))
        max_value = max(list(self.dic_movimientos.values()))

        if (min_value == 0) and (max_value == 0):
            dic_probs = {i: 1 / len(self.dic_movimientos) for i in range(1, n_movs + 1)}
        else:
            dic_probs = {i: self.dic_movimientos[i] - min_value for i in range(1, n_movs + 1)}
            S = sum(list(dic_probs.values()))
            dic_probs = {i: dic_probs[i] / S for i in range(1, n_movs + 1)}
        
        #print('ELEGIR NUEVA ARQUITECTURA')
        #print('     dic_probs', dic_probs)
        while True:

            # En base a estas probabilidades, elegir una opcion
            x = np.random.choice(list(dic_probs.keys()), p = list(dic_probs.values()))
            #x = 7 # fijado: cambiar!!
            print('\n\n     Nueva seleccion', x, dic_selecciones[x])
            if x == 1:
                # [ok] cambio 1 desarrollado: 1_agregar_neuronas
                str_arquitectura_new, dic_cambios = agregar_o_eliminar_neuronas(self.lista_n_neurs, self.str_arquitectura, 'agregar')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '1_agregar_neuronas', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break

            elif x == 2:
                # [ok] cambio 2: 2_eliminar_neuronas
                str_arquitectura_new, dic_cambios = agregar_o_eliminar_neuronas(self.lista_n_neurs, self.str_arquitectura, 'eliminar')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                
                try:
                    beta_inputs = self.ffnn.dic_betas
                except:
                    self.train(n_min)
                    beta_inputs = self.ffnn.dic_betas
                    
                dic_cambios['dic_betas'] = beta_inputs

                if dic_cambios['n_news'] != 0:
                    self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '2_eliminar_neuronas', dic_cambios = dic_cambios)
                else: # eliminar capa
                    str_arquitectura_new = ajustar_str_arquitectura_eliminar_capa(str_arquitectura_new)
                    self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '3_eliminar_capa', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 3:
                # cambio 3: 3_agregar_capa
                
                if len(self.str_arquitectura.split(',')) == max_capas_permitidas:
                    print(f'No se puede agregar capa porque la arquitectura actual es {self.str_arquitectura} y sse permite un máximo de {max_capas_permitidas} capas')
                    continue

                str_arquitectura_new, dic_cambios = agregar_nueva_capa(self.lista_n_neurs, self.str_arquitectura)
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '3_agregar_capa', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 4:
                # cambio 4: función siguiente
                str_arquitectura_new, dic_cambios = cambiar_funcion(self.str_arquitectura, 'siguiente')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '4_cambio_funcion', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 5:
                # cambio 5: función anterior
                str_arquitectura_new, dic_cambios = cambiar_funcion(self.str_arquitectura, 'anterior')
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '4_cambio_funcion', dic_cambios = dic_cambios)
                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
            
            elif x == 6:
                         
                str_arquitectura_new = self.str_arquitectura
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.campos_input_heredados = self.campos_input[:]
                dic_cambios = agregar_o_eliminar_inputs(self.campos_input, self.all_campos_inputs, 'agregar')
                if not dic_cambios['continuar']:
                    continue # Elegir un nuevo x, ya que no hay campos que puedan ser agregados
                self.campos_input = dic_cambios['campos_inputs_new']
                
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '6_cambios_inputs_agregar', dic_cambios = dic_cambios, campos_input_heredados = self.campos_input_heredados)

                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                
            elif x == 7:
                               
                str_arquitectura_new = self.str_arquitectura
                print('         CAMBIO ARQUITECTURA', x, self.str_arquitectura, str_arquitectura_new)
                self.campos_input_heredados = self.campos_input[:]
                try:
                    beta_inputs = self.ffnn.dic_betas[-1]
                except:
                    self.train(n_min)
                    beta_inputs = self.ffnn.dic_betas[-1]
                    
                dic_cambios = agregar_o_eliminar_inputs(self.campos_input, self.all_campos_inputs, 'eliminar', beta_inputs = beta_inputs)
                
                if not dic_cambios['continuar']:
                    continue # Elegir un nuevo x, ya que no hay campos que puedan ser agregados
                
                self.ffnn_new = Red_Neuronal(str_arquitectura_new, self.campos_input, self.cofre_base, seguimiento, self.output_level, self.metrica, self.df_activos, herencia = True, nombre_clase_heredada = self.ffnn.nombre, cambio = '7_cambios_inputs_eliminar', dic_cambios = dic_cambios, campos_input_heredados = self.campos_input_heredados)
                self.campos_input = self.ffnn_new.campos_input # Asignacion de campos_inputs nuevos a objeto Busqyeda inteligente              

                if self.ffnn_new.modo_arquitectura == 'nueva':
                    break
                                
        #print(x, str_arquitectura_new)
        self.str_arquitectura_anterior = self.str_arquitectura
        self.str_arquitectura = str_arquitectura_new # Se hace el cambio
        self.ultimo_cambio = x
        #sys.exit('Revision')
    
    def guardar(self):
        pickle_act(f'{self.cofre}{self.nombre}', variable = self, mode = 'save')
        return None
        
    def ejecutar(self, n_iters, n_min):
        
        entrenar_siguiente = True
        for i in range(n_iters):
            print(f'\n\n\n ######################################################### SIGUIENTE ITERACION {i} ######################################################################## \n\n\n')
            # 0. FFNN actual
            self.ffnn_actual()
            if i % 10 == 0:
                display(self.ffnn.df_info_ffnn.tail())
            # 1. Entrenar por una cantidad definida de epochs
            if entrenar_siguiente:
                self.train(n_min)
            # 1.5 Guardar ffnn actual 
            #\\\ ACTIVAR!!!!\\\
            #print('ACTIVAR!!!!')
            self.ffnn.guardar()
            # 2. Elegir una nueva arquitectura
            if self.inicial: # Si es la primera iteración de todas (solo cuando no existe nada, se guardan los parametros iniciales)
                self.best_test_loss_global = self.best_test_loss_iter # selected
                self.str_arquitectura_best = self.str_arquitectura
                self.best_model = self.ffnn.best_model # Se asigna el nuevo modelo encontrado (con todos sus parámetros) como mejor modelo
                self.mejor_red_neuronal = self.ffnn
                self.inicial = False
                self.elegir_nueva_arquitectura(n_min)
                #sys.exit('Salida para revision')
                continue
            
            # 3. Evaluar el delta en test ecm
            nuevo_delta = False
            if entrenar_siguiente:
                delta = self.best_test_loss_iter - self.best_test_loss_global
                self.dic_movimientos[self.ultimo_cambio] -= delta # cambio en asignacion de dic_movimientos
                nuevo_delta = True
            
            print('\n\n\n')
            print('nuevo_delta', nuevo_delta, 'delta', delta, 'best', self.best_test_loss_global, 'actual', self.best_test_loss_iter)
            
            # Se evalúa si es mejor
            if self.best_test_loss_iter < self.best_test_loss_global:
                print('\n\n Best test error mejora!!!')
                self.best_test_loss_global = self.best_test_loss_iter
                self.str_arquitectura_best = self.str_arquitectura
                self.best_model = self.ffnn.best_model # Se asigna el nuevo modelo encontrado (con todos sus parámetros) como mejor modelo
                self.mejor_red_neuronal = self.ffnn
                print('BASE NUEVA', self.str_arquitectura)
                # Se elige una nueva arquitectura
                self.elegir_nueva_arquitectura(n_min)
                entrenar_siguiente = True
            elif not entrenar_siguiente:
                self.elegir_nueva_arquitectura(n_min)
                entrenar_siguiente = True
            else: # de lo contrario, la base vuelve a ser el caso anterior
                self.str_arquitectura = self.str_arquitectura_anterior 
                print('BASE ANTERIOR', self.str_arquitectura)
                entrenar_siguiente = False
            
            #print('Continuar el 240520. Ver pagina 15')
            #print('Rescatar el error ACTUAL de la red (si existe previamente, de lo contrario, ocupar el error de la última iteracion antes de cambiar la arquitectura)')
            #print('Siempre guardar el best test error global')
            
            print('MEJOR RED NEURONAL HASTA AHORA')
            best_ffnn = self.mejor_red_neuronal
            print(best_ffnn.nombre)
            
            # 4. Guardar busqueda inteligente
            #print('GUARDAR (ACTIVAR)')
            self.guardar()

            print('OK')
        

# Ejecución

## Parámetros de configuración

In [37]:
metrica = 'Rendimiento'
n_min = 15 # n_min (cuantos epochs seguidos un entrenamiento debe pasar sin mejorar el best ECM para quebrar)
n_iters = 50 # número de iteraciones buscando arquitecturas diferentes
print('aumentar despues, por ahora solo de prueba')

aumentar despues, por ahora solo de prueba


In [38]:
lista_n_neurs = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96] # input (parametros)

## Ejecución

In [39]:
if 0 not in lista_n_neurs: # Configuracion de lista_n_neurs: Cuantas neuronas puede tener una capa cualquiera
    lista_n_neurs.append(0)

lista_n_neurs.sort()

In [40]:
for metrica in ['Rendimiento']:
    busqint = Busqueda_Inteligente('A1', metrica, output_level, cofre, lista_n_neurs)
    busqint.ejecutar(n_iters, n_min)

print('Fin')

campos_inputs_default en busq int
NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
     len 56
A1_Rendimiento_30.pkl
['A1_Rendimiento_30.pkl']
Rescate en ../Cofre/Busqueda_Inteligente/



 ######################################################### SIGUIENTE ITERACION 0 ######################################################################## 



tanh_8,sigmoid_32,sigmoid_8,tanh_4


,str_arquitectura,campos_input,campos_output,nombre
0,"tanh_8,sigmoid_32,sigmoid_8,tanh_4","Media_movil_10_Close,Media_movil_10_rendimient...",Y_Rendimiento_30,FFNN_1


NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
SIMBOLO NFLX 2002-05-23 00:00:00 2024-08-28 00:00:00
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
SIMBOLO FCEL 1992-06-25 00:00:00 2024-08-28 00:00:00
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
SIMBOLO DVA 1995-10-31 00:00:00 2024-08-28 00:00:00
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
SIMBOLO AAPL 1980-12-12 00:00:00 2024-08-28 00:00:00
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
SIMBOLO CAT 1962-01-02 00:00:00 2024-08-28 00:00:00
Y_Rendimiento_30 Y_Varianza_30
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 474us/step - accuracy: 0.0000e+00 - loss: 0.0014
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 540us/step - accuracy: 0.0000e+00 - loss: 0.0015
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 457us/step - accuracy: 0.0000e+00 - loss: 0.0014
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 444us/step - accuracy: 0.0000e+00 - loss: 0.0014
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 474us/step - accuracy: 0.0000e+00 - loss: 0.0014
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 445us/step - accuracy: 0.000

,str_arquitectura,campos_input,campos_output,nombre
2,"tanh_8,sigmoid_32,sigmoid_8,sigmoid_2","Media_movil_10_Close,Media_movil_10_rendimient...",Y_Rendimiento_30,FFNN_3
3,"tanh_8,sigmoid_32,sigmoid_8,relu_4,sigmoid_4","Media_movil_10_Close,Media_movil_10_rendimient...",Y_Rendimiento_30,FFNN_4
4,"tanh_8,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_10_rendimient...",Y_Rendimiento_30,FFNN_5
5,"tanh_8,sigmoid_32,tanh_8,sigmoid_4","Media_movil_10_Close,Media_movil_10_rendimient...",Y_Rendimiento_30,FFNN_6
6,"tanh_8,sigmoid_32,sigmoid_12,sigmoid_4","Media_movil_10_Close,Media_movil_10_rendimient...",Y_Rendimiento_30,FFNN_7


NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
SIMBOLO NFLX 2002-05-23 00:00:00 2024-08-28 00:00:00
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
SIMBOLO FCEL 1992-06-25 00:00:00 2024-08-28 00:00:00
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
SIMBOLO DVA 1995-10-31 00:00:00 2024-08-28 00:00:00
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
SIMBOLO AAPL 1980-12-12 00:00:00 2024-08-28 00:00:00
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
SIMBOLO CAT 1962-01-02 00:00:00 2024-08-28 00:00:00
Y_Rendimiento_30 Y_Varianza_30
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 442us/step - accuracy: 0.0000e+00 - loss: 0.0015
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 447us/step - accuracy: 0.0000e+00 - loss: 0.0015
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 484us/step - accuracy: 0.0000e+00 - loss: 0.0015
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 450us/step - accuracy: 0.0000e+00 - loss: 0.0015
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - accuracy: 0.0000e+00 - loss: 0.0015
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - accuracy: 0.000

...

NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
SIMBOLO NFLX 2002-05-23 00:00:00 2024-08-28 00:00:00
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
SIMBOLO FCEL 1992-06-25 00:00:00 2024-08-28 00:00:00
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
SIMBOLO DVA 1995-10-31 00:00:00 2024-08-28 00:00:00
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
SIMBOLO AAPL 1980-12-12 00:00:00 2024-08-28 00:00:00
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
SIMBOLO CAT 1962-01-02 00:00:00 2024-08-28 00:00:00
Y_Rendimiento_30 Y_Varianza_30
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 428us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 448us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 442us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 428us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 488us/step - accuracy: 0.000

,str_arquitectura,campos_input,campos_output,nombre
12,"tanh_8,relu_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_13
13,"tanh_4,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_14
14,"tanh_12,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_15
15,"sigmoid_8,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_16
16,"tanh_8,sigmoid_32,sigmoid_8,sigmoid_8","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_17


NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
SIMBOLO NFLX 2002-05-23 00:00:00 2024-08-28 00:00:00
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
SIMBOLO FCEL 1992-06-25 00:00:00 2024-08-28 00:00:00
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
SIMBOLO DVA 1995-10-31 00:00:00 2024-08-28 00:00:00
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
SIMBOLO AAPL 1980-12-12 00:00:00 2024-08-28 00:00:00
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
SIMBOLO CAT 1962-01-02 00:00:00 2024-08-28 00:00:00
Y_Rendimiento_30 Y_Varianza_30
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 455us/step - accuracy: 0.0000e+00 - loss: 0.0017
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 449us/step - accuracy: 0.0000e+00 - loss: 0.0017
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 450us/step - accuracy: 0.0000e+00 - loss: 0.0017
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 463us/step - accuracy: 0.0000e+00 - loss: 0.0017
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - accuracy: 0.0000e+00 - loss: 0.0017
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - accuracy: 0.000

,str_arquitectura,campos_input,campos_output,nombre
17,"tanh_8,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_18
18,"tanh_12,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_19
19,"tanh_8,relu_8,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_20
20,"relu_4,tanh_8,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_10_Close,Media_movil_1_Close,Media...",Y_Rendimiento_30,FFNN_21
21,"tanh_8,sigmoid_32,sigmoid_8,sigmoid_4","Media_movil_1_Close,Media_movil_1_rendimiento,...",Y_Rendimiento_30,FFNN_22


NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
SIMBOLO NFLX 2002-05-23 00:00:00 2024-08-28 00:00:00
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
SIMBOLO FCEL 1992-06-25 00:00:00 2024-08-28 00:00:00
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
SIMBOLO DVA 1995-10-31 00:00:00 2024-08-28 00:00:00
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
SIMBOLO AAPL 1980-12-12 00:00:00 2024-08-28 00:00:00
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
SIMBOLO CAT 1962-01-02 00:00:00 2024-08-28 00:00:00
Y_Rendimiento_30 Y_Varianza_30
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 441us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 458us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 467us/step - accuracy: 0.0000e+00 - loss: 0.0016
433/433 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 0.000

# Proceso Predict

In [41]:
# Elegir el mejor modelo

for metrica in ['Rendimiento']:
    busqint = Busqueda_Inteligente('A1', metrica, output_level, cofre, lista_n_neurs)
    print(busqint.nombre)
    best_ffnn = busqint.mejor_red_neuronal
    print('MEJOR RED NEURONAL ID', best_ffnn.nombre)
    df_x_predict, x_predict = best_ffnn.matrices_predict()
    display(df_x_predict)
    y_predict = best_ffnn.predict_model(x_predict)
    
    df_x_predict['y'] = y_predict.flatten()
    
    display(df_x_predict)
    print('Aqui desnormalizar')
    df_norm = best_ffnn.df_normalizacion.copy()
    df_norm = df_norm[df_norm['CAMPO'] == f'Y_{metrica}_{output_level}']
    y_min, y_max = df_norm['MIN'].values[0], df_norm['MAX'].values[0]
    df_y_predict = df_x_predict[['VALOR', 'DATE', 'y']]
    df_y_predict['y'] = df_y_predict['y'] * (y_max - y_min) + y_min

    max_date = df_y_predict['DATE'].max()
    dia_proyeccion = max_date + dt.timedelta(days = 30)
    df_y_predict = df_y_predict[df_y_predict['DATE'] == max_date].reset_index(drop = True)
    df_y_predict.to_csv(f'{cofre}Inputs_mkw/Rendimiento_{output_level}.csv', sep = ';', decimal = ',', index = False)
    display(df_y_predict)
        
    

campos_inputs_default en busq int
NFLX.pkl ../Cofre/Valor/
../Cofre/Valor/NFLX.pkl
FCEL.pkl ../Cofre/Valor/
../Cofre/Valor/FCEL.pkl
DVA.pkl ../Cofre/Valor/
../Cofre/Valor/DVA.pkl
AAPL.pkl ../Cofre/Valor/
../Cofre/Valor/AAPL.pkl
CAT.pkl ../Cofre/Valor/
../Cofre/Valor/CAT.pkl
     len 56
A1_Rendimiento_30.pkl
['A1_Rendimiento_30.pkl']
Rescate en ../Cofre/Busqueda_Inteligente/
A1_Rendimiento_30
MEJOR RED NEURONAL ID FFNN_44


NAME,VALOR,DATE,Media_movil_10_rendimiento,Media_movil_1_rendimiento,Media_movil_20_rendimiento,Media_movil_2_Close,Media_movil_2_rendimiento,Media_movil_3_Close,Media_movil_3_rendimiento,Media_movil_multiplicativa_10_rendimiento,...,Polinomio_rendimiento_3_10_0.2,Polinomio_rendimiento_3_10_0.3,Polinomio_rendimiento_5_10_0.2,Polinomio_rendimiento_5_10_0.3,Polinomio_rendimiento_5_30_0.2,Polinomio_rendimiento_5_30_0.3,Suavizamiento_Exponencial_0.1_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.5_rendimiento
0,AAPL,2024-06-24,0.310470,0.311667,0.479611,0.880770,0.261894,0.881189,0.260292,0.097370,...,0.081304,0.078012,0.019607,0.020370,0.269677,0.273142,0.336229,0.280021,0.886204,0.291581
1,AAPL,2024-06-25,0.313398,0.311667,0.479442,0.881688,0.261893,0.882108,0.265290,0.098200,...,0.081268,0.077981,0.019596,0.020361,0.269626,0.273104,0.336220,0.281816,0.885934,0.292944
2,AAPL,2024-06-26,0.312734,0.313494,0.478507,0.884118,0.263474,0.884036,0.266780,0.098013,...,0.081106,0.077830,0.019554,0.020321,0.269437,0.272925,0.337778,0.284783,0.887778,0.295387
3,AAPL,2024-06-27,0.317020,0.321774,0.486057,0.894946,0.272224,0.891569,0.275026,0.099209,...,0.081074,0.077799,0.019544,0.020313,0.269390,0.272889,0.346277,0.294607,0.897593,0.304587
4,AAPL,2024-06-28,0.316229,0.313237,0.483714,0.905606,0.272002,0.899997,0.276307,0.098986,...,0.081206,0.077930,0.019574,0.020347,0.269526,0.273039,0.346609,0.293495,0.904309,0.300961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,NFLX,2024-07-19,0.299736,0.307470,0.446192,0.925489,0.251546,0.931710,0.254998,0.094292,...,0.093061,0.090171,0.018823,0.022448,0.265599,0.282190,0.316229,0.275484,0.932150,0.286507
176,NFLX,2024-07-20,0.294939,0.303066,0.442163,0.915363,0.250814,0.920712,0.248521,0.092934,...,0.093009,0.090126,0.018797,0.022442,0.265473,0.282165,0.310847,0.270593,0.922089,0.282120
177,NFLX,2024-07-21,0.301060,0.315086,0.444415,0.911791,0.257407,0.916214,0.257637,0.094658,...,0.092941,0.090065,0.018783,0.022430,0.265408,0.282112,0.316306,0.278416,0.920457,0.291508
178,NFLX,2024-07-22,0.315097,0.315056,0.446652,0.918561,0.267787,0.916089,0.263827,0.098675,...,0.092872,0.090000,0.018772,0.022413,0.265357,0.282044,0.321195,0.283864,0.923040,0.296174


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


NAME,VALOR,DATE,Media_movil_10_rendimiento,Media_movil_1_rendimiento,Media_movil_20_rendimiento,Media_movil_2_Close,Media_movil_2_rendimiento,Media_movil_3_Close,Media_movil_3_rendimiento,Media_movil_multiplicativa_10_rendimiento,...,Polinomio_rendimiento_3_10_0.3,Polinomio_rendimiento_5_10_0.2,Polinomio_rendimiento_5_10_0.3,Polinomio_rendimiento_5_30_0.2,Polinomio_rendimiento_5_30_0.3,Suavizamiento_Exponencial_0.1_rendimiento,Suavizamiento_Exponencial_0.3_rendimiento,Suavizamiento_Exponencial_0.5_Close,Suavizamiento_Exponencial_0.5_rendimiento,y
0,AAPL,2024-06-24,0.310470,0.311667,0.479611,0.880770,0.261894,0.881189,0.260292,0.097370,...,0.078012,0.019607,0.020370,0.269677,0.273142,0.336229,0.280021,0.886204,0.291581,0.554637
1,AAPL,2024-06-25,0.313398,0.311667,0.479442,0.881688,0.261893,0.882108,0.265290,0.098200,...,0.077981,0.019596,0.020361,0.269626,0.273104,0.336220,0.281816,0.885934,0.292944,0.554344
2,AAPL,2024-06-26,0.312734,0.313494,0.478507,0.884118,0.263474,0.884036,0.266780,0.098013,...,0.077830,0.019554,0.020321,0.269437,0.272925,0.337778,0.284783,0.887778,0.295387,0.554113
3,AAPL,2024-06-27,0.317020,0.321774,0.486057,0.894946,0.272224,0.891569,0.275026,0.099209,...,0.077799,0.019544,0.020313,0.269390,0.272889,0.346277,0.294607,0.897593,0.304587,0.552987
4,AAPL,2024-06-28,0.316229,0.313237,0.483714,0.905606,0.272002,0.899997,0.276307,0.098986,...,0.077930,0.019574,0.020347,0.269526,0.273039,0.346609,0.293495,0.904309,0.300961,0.553115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,NFLX,2024-07-19,0.299736,0.307470,0.446192,0.925489,0.251546,0.931710,0.254998,0.094292,...,0.090171,0.018823,0.022448,0.265599,0.282190,0.316229,0.275484,0.932150,0.286507,0.557061
176,NFLX,2024-07-20,0.294939,0.303066,0.442163,0.915363,0.250814,0.920712,0.248521,0.092934,...,0.090126,0.018797,0.022442,0.265473,0.282165,0.310847,0.270593,0.922089,0.282120,0.557636
177,NFLX,2024-07-21,0.301060,0.315086,0.444415,0.911791,0.257407,0.916214,0.257637,0.094658,...,0.090065,0.018783,0.022430,0.265408,0.282112,0.316306,0.278416,0.920457,0.291508,0.556296
178,NFLX,2024-07-22,0.315097,0.315056,0.446652,0.918561,0.267787,0.916089,0.263827,0.098675,...,0.090000,0.018772,0.022413,0.265357,0.282044,0.321195,0.283864,0.923040,0.296174,0.555038


Aqui desnormalizar


NAME,VALOR,DATE,y
0,AAPL,2024-07-23,0.000830
1,CAT,2024-07-23,-0.000281
2,DIS,2024-07-23,0.000215
3,DVA,2024-07-23,-0.000051
4,FCEL,2024-07-23,0.006320
5,NFLX,2024-07-23,0.000578


In [42]:
sys.exit()

SystemExit: 

# Otros (revision etapa predit)

In [ ]:
df_raw_info = pd.DataFrame()
for i in range(len(df_activos)):
    simbolo, nombre = df_activos.loc[i]
    valor = Valor(simbolo, nombre, cofre) # Si no existe el objeto, se crea 
    if len(valor.raw_x) == 0: # No hay datos que aportar
        continue
    #print(simbolo, nombre, len(valor.raw_x))
    new_raw_x = valor.raw_x.copy()
    new_raw_x['VALOR'] = simbolo
    df_raw_info = pd.concat([df_raw_info, new_raw_x], axis = 0)

df_raw_info = df_raw_info[['VALOR', 'DATE', 'NAME', 'X']]
df_raw_info

# Primero, se separan los inputs de los outputs
df_raw_info['NATURALEZA'] = np.where(df_raw_info['NAME'].str[:2] == 'Y_', 'Y', 'X')
df_raw_y = df_raw_info[df_raw_info['NATURALEZA'] == 'Y'].reset_index(drop = True)
df_raw_x = df_raw_info[df_raw_info['NATURALEZA'] == 'X'].reset_index(drop = True)

df_X = df_raw_x.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()
df_Y = df_raw_y.pivot_table(index = ['VALOR', 'DATE'], columns = 'NAME', values = 'X').reset_index()

df = df_X.merge(df_Y, on = ['VALOR', 'DATE'], how = 'outer')

while True:
    aprobado = True
    for c in list(set(df.columns) - {'DATE', 'VALOR'}):
        if len(df[df[c].isna()]) > 0:
            aprobado = False
    if aprobado:
        break
    print('A. Espera de completitud de valores en Valor.py: Espera de 30s.')
    time.sleep(30)
            

# Limpieza de df
lista_campos_output = list(set(df_Y.columns) - {'DATE', 'VALOR'})
set_delta_dates = set()
for c in lista_campos_output:

    delta = c.split('_')[-1]
    set_delta_dates.add(delta)
        

print(f'Y_Rendimiento_{delta}', f'Y_Varianza_{delta}')  
          
df_predict = df[((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se aislan los datos para después hacer el predict
if len(df_predict) == 0:
    sys.exit('Predict sin datos')
else:
    None
    #print(self.nombre)
    #print(self.df_predict)
df = df[~((df[f'Y_Rendimiento_{delta}'] == 0) & (df[f'Y_Varianza_{delta}'] == 0))].reset_index(drop = True) # Se excluyen los casos en los que la varianza y el rend de un día, son 0, para el mismo delta
df_predict